# Surrogate training
Trains the U-Net surrogate, then evaluates it on the held-out validation set
and on an out-of-distribution (OOD) resonant waveguide geometry.

In [ ]:
import numpy as np
import random
import torch
from torch.utils.data import DataLoader, Subset
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.colors import BoundaryNorm, ListedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable
from torch import nn
from torch.nn import functional as F

from surrogate_model import UNet
from strain_dataset import StrainDataset


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Dataset and dataloaders

In [ ]:
NPZ = 'dataset_12mkm_2.npz'

ds_train = StrainDataset(NPZ, augment=False)
ds_eval = StrainDataset(NPZ, augment=False)

N = len(ds_eval)
n_train = int(0.8 * N)
n_val = int(0.1 * N)
n_test = N - n_train - n_val

g = torch.Generator().manual_seed(42)
perm = torch.randperm(N, generator=g).tolist()

train_idx = perm[:n_train]
val_idx = perm[n_train : n_train + n_val]
test_idx = perm[n_train + n_val:]

train_set = Subset(ds_train, train_idx)
val_set = Subset(ds_eval,  val_idx)
test_set = Subset(ds_eval,  test_idx)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=0)

# sanity check
print(f"train: {len(train_set)}, val: {len(val_set)}, test: {len(test_set)}")
print(f"train augment: {train_set.dataset.augment}")   # False
print(f"val   augment: {val_set.dataset.augment}")     # False

## Gradient loss term

In [ ]:
def gradient_loss(pred, target):
    pred_dx = pred[..., :, 1:] - pred[..., :, :-1]
    pred_dy = pred[..., 1:, :] - pred[..., :-1, :]

    tgt_dx = target[..., :, 1:] - target[..., :, :-1]
    tgt_dy = target[..., 1:, :] - target[..., :-1, :]

    return F.l1_loss(pred_dx, tgt_dx) + F.l1_loss(pred_dy, tgt_dy)

## Training loop

In [ ]:
model = UNet().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loaders = {"train": train_loader, "valid": val_loader}

max_epochs = 40
losses = {"train": [], "valid": []}

for epoch in range(max_epochs):
    for k, dataloader in loaders.items():
        epoch_loss = 0
        epoch_batches = 0

        for x_batch, y_batch in dataloader:
            if k == "train":
                model.train()
                optimizer.zero_grad()
                outp = model(x_batch.to(device))
                mse  = criterion(outp, y_batch.to(device))
                grad = gradient_loss(outp, y_batch.to(device))
                loss = 1000*mse + 10 * grad
                loss.backward()
                optimizer.step()
            else:
                model.eval()
                with torch.no_grad():
                    outp = model(x_batch.to(device))
                mse  = criterion(outp, y_batch.to(device))
                grad = gradient_loss(outp, y_batch.to(device))
                loss = 1000*mse + 10 * grad

            epoch_loss += loss.item()
            epoch_batches += 1

        avg_loss = epoch_loss / epoch_batches
        losses[k].append(avg_loss)

        if k == "train":
            print(f"Epoch: {epoch+1}")
        print(f"Loader: {k}. MSE Loss: {avg_loss:.6f}")

    plt.clf()
    plt.plot(losses["train"], label="train")
    plt.plot(losses["valid"], label="valid")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("Learning curves")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    # plt.savefig("learning_curves.png", dpi=100)
    plt.show()

## Validation inference and visualization

In [ ]:
model = UNet().to(device)
state_dict = torch.load('unet5_r2_0983_256_3007.pth', weights_only=True)
model.load_state_dict(state_dict)
model = model.to(device)

In [ ]:
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        all_preds.append(model(xb.to(device)).cpu())
        all_targets.append(yb)

all_preds   = torch.cat(all_preds,   dim=0)   # (N, 1, H, W)
all_targets = torch.cat(all_targets, dim=0)   # (N, 1, H, W)

mse_val = criterion(all_preds, all_targets).item()
mae_val = torch.mean(torch.abs(all_preds - all_targets)).item()

y_true = all_targets
y_pred = all_preds
ss_res = torch.sum((y_true - y_pred) ** 2)
ss_tot = torch.sum((y_true - y_true.mean()) ** 2)
r2_val = (1.0 - ss_res / ss_tot).item()

N_show = 12
indices = torch.randperm(len(all_preds))[:N_show]

ROWS, TRIPLETS_PER_ROW = 3, 4
NCOLS = TRIPLETS_PER_ROW * 3  # 6 columns

fig, axes = plt.subplots(ROWS, NCOLS, figsize=(NCOLS * 3.2, ROWS * 3.5))

for plot_idx, data_idx in enumerate(indices):
    row = plot_idx // TRIPLETS_PER_ROW
    t   = plot_idx  % TRIPLETS_PER_ROW
    c   = t * 3

    # unpack all 3 input channels from the dataset
    sample_x = val_loader.dataset[data_idx.item()][0]  # (3, H, W)
    inp_x  = sample_x[0].numpy()   # support map
    grid_x = sample_x[1].numpy()   # X coordinates (H, W)
    grid_y = sample_x[2].numpy()   # Y coordinates (H, W)

    # extent: [xmin, xmax, ymin, ymax] for imshow
    xmin, xmax = float(grid_x.min()), float(grid_x.max())
    ymin, ymax = float(grid_y.min()), float(grid_y.max())
    extent = [xmin, xmax, ymin, ymax]

    inp = all_preds[data_idx, 0].numpy()
    gt  = all_targets[data_idx, 0].numpy()
    vmin, vmax = gt.min(), gt.max()

    # input (support map) with coordinate axes
    axes[row, c].imshow(inp_x, cmap="gray_r", origin="lower", extent=extent, aspect="auto")
    axes[row, c].set_title(f"Input #{data_idx+1}", fontsize=9)
    axes[row, c].set_xlabel("X, μm", fontsize=7)
    axes[row, c].set_ylabel("Y, μm", fontsize=7)
    axes[row, c].tick_params(labelsize=6)

    # prediction with coordinate axes
    im1 = axes[row, c+1].imshow(inp, cmap="coolwarm", origin="lower",
                                 vmin=vmin, vmax=vmax, extent=extent, aspect="auto")
    axes[row, c+1].set_title(f"Prediction #{data_idx+1}", fontsize=9)
    axes[row, c+1].set_xlabel("X, μm", fontsize=7)
    axes[row, c+1].set_ylabel("Y, μm", fontsize=7)
    axes[row, c+1].tick_params(labelsize=6)
    plt.colorbar(im1, ax=axes[row, c+1], fraction=0.046)

    # ground truth with coordinate axes
    im2 = axes[row, c+2].imshow(gt, cmap="coolwarm", origin="lower",
                                 vmin=vmin, vmax=vmax, extent=extent, aspect="auto")
    axes[row, c+2].set_title(f"Ground Truth #{data_idx+1}", fontsize=9)
    axes[row, c+2].set_xlabel("X, μm", fontsize=7)
    axes[row, c+2].set_ylabel("Y, μm", fontsize=7)
    axes[row, c+2].tick_params(labelsize=6)
    plt.colorbar(im2, ax=axes[row, c+2], fraction=0.046)

plt.suptitle(
    f"Validation inference  |  "
    f"MSE = {mse_val:.6f}, MAE = {mae_val:.6f}, R² = {r2_val:.4f}",
    fontsize=13
)
plt.tight_layout()
# plt.savefig("inference_val_results.png", dpi=120, bbox_inches="tight")
plt.show()

## Out-of-distribution (OOD) inference helper

In [ ]:
def run_inference(model, support_map, Lx, Ly, device, pad_multiple=16, cmap_strain='coolwarm'):
    H, W = support_map.shape

    # coordinates from 0 to Lx/Ly, matching the dataset convention
    xs = torch.linspace(0, Lx, W)
    ys = torch.linspace(0, Ly, H)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')

    supp_t = torch.tensor(support_map, dtype=torch.float32).unsqueeze(0)
    grid_x = grid_x.unsqueeze(0)
    grid_y = grid_y.unsqueeze(0)
    x = torch.cat([supp_t, grid_x, grid_y], dim=0).unsqueeze(0)  # (1,3,H,W)

    pad_h = (pad_multiple - H % pad_multiple) % pad_multiple
    pad_w = (pad_multiple - W % pad_multiple) % pad_multiple
    x_padded = F.pad(x, (0, pad_w, 0, pad_h))

    model.eval()
    with torch.no_grad():
        pred_padded = model(x_padded.to(device)).cpu()
    pred = pred_padded[0, 0, :H, :W].numpy()

    ext = [0, Lx, 0, Ly]
    # ... visualization unchanged
    return pred, grid_x.squeeze().numpy(), grid_y.squeeze().numpy()

## OOD comparison: FDM vs. surrogate on the resonant waveguide geometry

In [ ]:
struct     = np.load('swg_256.npz')[next(iter(np.load('swg_256.npz').keys()))]
strain_fdm = np.load('strain_swg_256_2.npz')[next(iter(np.load('strain_swg_256_2.npz').keys()))] * 100

In [ ]:
Lx, Ly = 12, 12

pred, gx, gy = run_inference(model, struct, Lx=Lx, Ly=Ly, device=device)
strain_nn = pred
residual  = strain_nn - strain_fdm

In [ ]:
def load_first(path):
    d = np.load(path)
    return d[next(iter(d.keys()))]

# TODO: set the paths to the OOD geometry / FDM strain reference
struct     = load_first('swg_256.npz')
strain_fdm = load_first('strain_swg_256_2.npz') * 100
strain_fdm = strain_fdm
strain_nn  = pred
residual   = strain_nn - strain_fdm
ext = [0, 12, 0, 12]   # μm

binary_cmap = ListedColormap(['white', 'black'])
binary_norm = BoundaryNorm([0, 0.5, 1], binary_cmap.N)

fmt = matplotlib.ticker.FuncFormatter(lambda v, _: f'{v:g}')

def set_minmax_ticks(ax, ext):
    x0, x1, y0, y1 = ext
    ax.set_xticks([x0, x1])
    ax.set_yticks([y0, y1])
    ax.xaxis.set_major_formatter(fmt)
    ax.yaxis.set_major_formatter(fmt)

def add_colorbar(ax, mappable, label, ticks=None):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.04)
    cb = plt.colorbar(mappable, cax=cax)
    cb.set_label(label, fontsize=6)
    if ticks is not None:
        cb.set_ticks(ticks)
    cb.ax.tick_params(labelsize=5.5, width=0.5, length=2)
    cb.outline.set_linewidth(0.5)
    return cb

def panel_label(ax, letter):
    ax.text(-0.25, 1.06, letter, transform=ax.transAxes,
            fontsize=8, fontweight='bold', color='black',
            va='top', ha='left')

mm = 1 / 25.4
fig, axes = plt.subplots(1, 4, figsize=(180*mm, 64*mm), constrained_layout=False)
fig.subplots_adjust(left=0.07, right=0.97, top=0.88, bottom=0.18, wspace=0.65)

ax = axes[0]
im = ax.imshow(struct, cmap=binary_cmap, norm=binary_norm,
               origin='lower', extent=ext, aspect='equal')
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.04)
cb = plt.colorbar(im, cax=cax, ticks=[0, 1])
cb.set_label('Material', fontsize=6)
cb.ax.set_yticklabels(['0', '1'])
cb.ax.tick_params(labelsize=5.5, width=0.5, length=2)
cb.outline.set_linewidth(0.5)
panel_label(ax, 'b')
ax.set_title('Structure', fontsize=7, pad=3)
ax.set_xlabel('μm', fontsize=6)
ax.set_ylabel('μm', fontsize=6)
ax.tick_params(top=False, right=False)
set_minmax_ticks(ax, ext)

ax = axes[1]
vmax_fdm = np.abs(strain_fdm).max()
im = ax.imshow(strain_fdm, cmap='coolwarm', origin='lower',
               extent=ext, aspect='equal', vmin=-vmax_fdm, vmax=vmax_fdm)
add_colorbar(ax, im, 'Strain (FDM)')
panel_label(ax, 'c')
ax.set_title('FDM strain', fontsize=7, pad=3)
ax.set_xlabel('μm', fontsize=6)
ax.tick_params(top=False, right=False, labelleft=False)
set_minmax_ticks(ax, ext)

ax = axes[2]
im = ax.imshow(strain_nn, cmap='coolwarm', origin='lower',
               extent=ext, aspect='equal', vmin=-vmax_fdm, vmax=vmax_fdm)
add_colorbar(ax, im, 'Strain (NN)')
panel_label(ax, 'd')
ax.set_title('NN strain', fontsize=7, pad=3)
ax.set_xlabel('μm', fontsize=6)
ax.tick_params(top=False, right=False, labelleft=False)
set_minmax_ticks(ax, ext)

ax = axes[3]
vmax_res = np.abs(residual).max()
im = ax.imshow(residual, cmap='coolwarm', origin='lower',
               extent=ext, aspect='equal', vmax=vmax_res)
add_colorbar(ax, im, 'Residual')
panel_label(ax, 'e')
ax.set_title('NN − FDM', fontsize=7, pad=3)
ax.set_xlabel('μm', fontsize=6)
ax.tick_params(top=False, right=False, labelleft=False)
set_minmax_ticks(ax, ext)

# plt.savefig('figure_strain_comparison_coolwarm.pdf', bbox_inches='tight')
# plt.savefig('figure_strain_comparison_coolwarm.png', dpi=300, bbox_inches='tight')
# plt.show()